# 04 — OULAD leakage-safe early-warning evaluation

This notebook evaluates an early-warning version of the enrollment-risk task. The target remains the final course outcome (`Fail`/`Withdrawn` versus `Pass`/`Distinction`), but each feature matrix is built as if the planner stopped observing the course at a specified course-relative day.

The evaluation uses three explicit safeguards:

1. assessment submissions and VLE activity after the cutoff are excluded;
2. future `date_unregistration` is excluded from cutoff features; and
3. the latest course presentation is held out as test data, so later presentations are not used to train earlier predictions.

This is an evaluation of early-warning feasibility, not a claim that a deployed planner would know the final outcome at prediction time.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_oulad_tables
from src.data.cleaner import clean_oulad_tables
from src.features.feature_engineering import build_oulad_features
from src.models.baseline import (
    MODEL_NAMES,
    split_time_aware_data,
    train_time_aware_baselines,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'oulad'
CUTOFFS = [30, 60, 90]
tables = clean_oulad_tables(load_oulad_tables(DATA_DIR))
print(f'Loaded cleaned OULAD tables from {DATA_DIR}')

## Observation availability

The counts below show how much dated information exists by cutoff. Missing dates are not treated as observed evidence.

In [ ]:
availability = []
for cutoff in CUTOFFS:
    availability.append({
        'cutoff_days': cutoff,
        'assessment_submissions_observed': int((tables['studentAssessment']['date_submitted'].notna() & (tables['studentAssessment']['date_submitted'] <= cutoff)).sum()),
        'vle_rows_observed': int((tables['studentVle']['date'].notna() & (tables['studentVle']['date'] <= cutoff)).sum()),
        'registrations_with_future_unregistration_excluded': int(tables['studentRegistration']['date_unregistration'].notna().sum()),
    })
availability_df = pd.DataFrame(availability)
display(availability_df.style.background_gradient(cmap='YlGnBu'))

## Four-model time-aware evaluation

At each cutoff, the feature builder is rerun from the cleaned tables. The four baselines are logistic regression, random forest, extra-trees, and histogram gradient boosting. The model ranking is based on ROC-AUC, then balanced accuracy, then F1.

In [ ]:
evaluation_rows = []
fitted_by_cutoff = {}
split_by_cutoff = {}
for cutoff in CUTOFFS:
    features = build_oulad_features(tables, clean=False, cutoff_days=cutoff)
    assert 'date_unregistration' not in features.columns
    best_name, fitted, model_results = train_time_aware_baselines(features)
    train_features, test_features, train_target, test_target = split_time_aware_data(features)
    fitted_by_cutoff[cutoff] = fitted
    split_by_cutoff[cutoff] = (test_features, test_target)
    for model_name in MODEL_NAMES:
        row = {'cutoff_days': cutoff, 'model': model_name, 'selected': model_name == best_name}
        row.update(model_results[model_name])
        evaluation_rows.append(row)

results = pd.DataFrame(evaluation_rows)
display(results.sort_values(['cutoff_days', 'roc_auc'], ascending=[True, False]).style.background_gradient(cmap='viridis', subset=['roc_auc', 'balanced_accuracy', 'f1']))

In [ ]:
presentation_summary = []
for cutoff, (test_features, _) in split_by_cutoff.items():
    presentation_summary.append({
        'cutoff_days': cutoff,
        'train_presentations': ', '.join(sorted(set(tables['studentInfo']['code_presentation']) - set(test_features['code_presentation']))),
        'test_presentations': ', '.join(sorted(test_features['code_presentation'].unique())),
        'test_rows': len(test_features),
    })
display(pd.DataFrame(presentation_summary))
print('The default split holds out the latest available presentation at every cutoff.')

## Colorful metric comparison

These plots show how performance changes as more course-relative observations become available. They should be read alongside the time-aware split and not compared directly with the retrospective full-course benchmark.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
colors = {'logistic_regression': '#264653', 'random_forest': '#2a9d8f', 'extra_trees': '#e9c46a', 'hist_gradient_boosting': '#e76f51'}
for axis, metric in zip(axes, ['roc_auc', 'balanced_accuracy', 'f1']):
    for model_name in MODEL_NAMES:
        subset = results[results['model'] == model_name].sort_values('cutoff_days')
        axis.plot(subset['cutoff_days'], subset[metric], marker='o', linewidth=2, label=model_name.replace('_', ' ').title(), color=colors[model_name])
    axis.set_title(metric.replace('_', ' ').title())
    axis.set_xlabel('Observed course-relative days')
    axis.set_ylim(0, 1.02)
    axis.grid(alpha=0.25)
axes[0].set_ylabel('Score')
axes[-1].legend(loc='lower right', fontsize=8)
fig.suptitle('Leakage-safe early-warning performance')
plt.show()

## Confusion matrices at the 60-day cutoff

Confusion matrices make the operational trade-off visible: false negatives are learners at risk that the planner failed to flag, while false positives are learners flagged for support who ultimately were not in the risk class.

In [ ]:
focus_cutoff = 60
test_features, test_target = split_by_cutoff[focus_cutoff]
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
for axis, model_name in zip(axes.ravel(), MODEL_NAMES):
    ConfusionMatrixDisplay.from_estimator(
        fitted_by_cutoff[focus_cutoff][model_name],
        test_features,
        test_target,
        display_labels=['Pass/Distinction', 'Fail/Withdrawn'],
        cmap='Blues',
        ax=axis,
    )
    axis.set_title(model_name.replace('_', ' ').title())
fig.suptitle('60-day time-aware confusion matrices')
plt.show()

## Interpretation and limitations

The selected model and scores must be taken from the executed result table or the reproducible JSON output generated by `pipeline/early_warning_pipeline.py`; this notebook does not hard-code metrics.

This design still uses the final course outcome as the label, so it measures retrospective discrimination under an early-observation constraint. A production study planner would need prospective validation, intervention-aware evaluation, calibration and threshold selection, and a policy for handling learners with too little observed activity.